In [ ]:
import torch
import matplotlib.pyplot as plt

BASE_PATH = "/home/alex/internship/GradientDistillation"
run_name = "local_seathru_test_all_visu"
data = torch.load(
    f"{BASE_PATH}/logged_files/distillation/aqua20/dinov2_vitb/{run_name}/data.pth",
    weights_only=False, map_location="cpu",
)
snapshots = data["pyramid_snapshots"]   # liste de dicts {"I", "J", "T", "d", "beta", "B", "J_levels", "level_res"}
steps = data["pyramid_snapshot_steps"]
print(f"{len(snapshots)} snapshots, steps: {steps}")
print(f"clés d'un snapshot: {snapshots[0].keys()}")
print(f"shape de I: {snapshots[0]['I'].shape}")

In [ ]:
def show_evolution(snapshots, steps, key="I", class_indices=None, figsize_per_img=1.5):
    """key: 'I', 'J', 'T' (3 canaux) ou 'd' (1 canal, affiché en colormap)."""
    N = snapshots[0][key].shape[0]
    cols = list(class_indices) if class_indices is not None else list(range(N))

    n_rows, n_cols = len(snapshots), len(cols)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * figsize_per_img, n_rows * figsize_per_img),
                             squeeze=False)
    for i, (snap, step) in enumerate(zip(snapshots, steps)):
        imgs = snap[key]
        for j, c in enumerate(cols):
            if imgs.shape[1] == 1:  # depth map
                axes[i][j].imshow(imgs[c, 0].cpu(), cmap="viridis")
            else:
                axes[i][j].imshow(imgs[c].clamp(0, 1).permute(1, 2, 0).cpu())
            axes[i][j].axis("off")
        axes[i][0].set_ylabel(f"it {step}", rotation=0, labelpad=30)
        axes[i][0].axis("on")
        axes[i][0].set_xticks([]); axes[i][0].set_yticks([])
    fig.suptitle(key)
    plt.tight_layout()
    plt.show()

show_evolution(snapshots, steps, key="I")
show_evolution(snapshots, steps, key="J")
show_evolution(snapshots, steps, key="T")
show_evolution(snapshots, steps, key="d")

In [ ]:
def show_pyramid_evolution(snapshots, steps, class_idx=0, key="J_levels", figsize_per_img=1.8):
    max_levels = max(len(s[key]) for s in snapshots)
    n_rows = len(snapshots)
    fig, axes = plt.subplots(n_rows, max_levels,
                             figsize=(max_levels * figsize_per_img, n_rows * figsize_per_img),
                             squeeze=False)
    for i, (snap, step) in enumerate(zip(snapshots, steps)):
        for k in range(max_levels):
            ax = axes[i][k]
            if k < len(snap[key]):
                ax.imshow(snap[key][k][class_idx].clamp(0, 1).permute(1, 2, 0).cpu())
                if i == 0 or k == len(snap[key]) - 1:
                    ax.set_title(f"≤ {snap['level_res'][k]}²", fontsize=7)
            ax.axis("off")
        axes[i][0].set_ylabel(f"it {step}", rotation=0, labelpad=30)
        axes[i][0].axis("on")
        axes[i][0].set_xticks([]); axes[i][0].set_yticks([])
    fig.suptitle(f"J cumulatif — classe {class_idx}")
    plt.tight_layout()
    plt.show()

show_pyramid_evolution(snapshots, steps, class_idx=1)

In [ ]:
def show_B_evolution(snapshots, steps, class_idx=0):
    B = torch.stack([s["B"][class_idx, :, 0, 0] for s in snapshots])     # (n_steps, 3)
    beta = torch.stack([s["beta"][class_idx, :, 0, 0] for s in snapshots])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
    for c, col in zip(range(3), ["red", "green", "blue"]):
        ax1.plot(steps, B[:, c], color=col, marker="o", label=f"B_{col[0].upper()}")
        ax2.plot(steps, beta[:, c], color=col, marker="o", label=f"β_{col[0].upper()}")
    ax1.set_title(f"B — classe {class_idx}"); ax1.legend(); ax1.set_xlabel("iteration")
    ax2.set_title(f"β — classe {class_idx}"); ax2.legend(); ax2.set_xlabel("iteration")
    plt.tight_layout()
    plt.show()

show_B_evolution(snapshots, steps, class_idx=0)

In [ ]:
def show_B_evolution_grid(snapshots, steps, key="B", n_cols=5, class_names=None):
    """Grille de courbes pour toutes les classes. key: 'B' ou 'beta'."""
    N = snapshots[0][key].shape[0]
    vals = torch.stack([s[key][:, :, 0, 0] for s in snapshots])  # (n_steps, N, 3)

    n_rows = (N + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(3 * n_cols, 2.2 * n_rows),
                             sharex=True, sharey=True, squeeze=False)
    for idx in range(n_rows * n_cols):
        ax = axes[idx // n_cols][idx % n_cols]
        if idx >= N:
            ax.axis("off")
            continue
        for c, col in zip(range(3), ["red", "green", "blue"]):
            ax.plot(steps, vals[:, idx, c], color=col, lw=1.2)
        title = class_names[idx] if class_names is not None else f"classe {idx}"
        ax.set_title(title, fontsize=8)
        ax.tick_params(labelsize=6)

    fig.suptitle(key, fontsize=12)
    fig.supxlabel("iteration", fontsize=9)
    plt.tight_layout()
    plt.show()

show_B_evolution_grid(snapshots, steps, key="B")
show_B_evolution_grid(snapshots, steps, key="beta")

In [ ]:
beta_final = snapshots[-1]["beta"][:, :, 0, 0]  # (N, 3)
ratio = beta_final[:, 2] / beta_final[:, 1]      # β_B / β_G : >1 = "côtier" (vert pénètre mieux)
plt.figure(figsize=(8, 3))
plt.bar(range(len(ratio)), ratio)
plt.axhline(1.0, color="k", ls="--", lw=0.8)
plt.xlabel("classe"); plt.ylabel("β_B / β_G")
plt.title("Ordering spectral final par classe (>1 : eau côtière, <1 : eau océanique)")
plt.tight_layout(); plt.show()

In [ ]:

import os


class_names = sorted(os.listdir("/home/alex/internship/datasets/aqua20/data/aqua20/train"))  # à adapter
plt.figure(figsize=(9, 3.2))
plt.bar(range(len(ratio)), ratio)
plt.axhline(1.0, color="k", ls="--", lw=0.8)
plt.xticks(range(len(ratio)), class_names, rotation=60, ha="right", fontsize=7)
plt.ylabel("β_B / β_G")
plt.tight_layout(); plt.show()